In [1]:
import os
import warnings
import seaborn as sns
# Suppress warnings
warnings.filterwarnings('ignore')

In [2]:
# Essential libraries
import numpy as np
from matplotlib import pyplot as plt
from scipy.stats import spearmanr, skew
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from catboost import CatBoostRegressor
import torch
import pandas as pd
import numpy as np
import scipy.stats as stats

In [3]:
# Set random seed for reproducibility
seed = 69
torch.manual_seed(seed)
np.random.seed(seed)

In [4]:
# Define root directory
root = '.'

In [5]:
data = pd.read_csv('./new_dataset/maison-llf-features.CSV', sep=",")

In [7]:
data.head(3)

,participant,timestamp,clinical-timestamp,sis-01,sis-02,sis-03,sis-04,sis-05,sis-06,sis,...,sleep-duration-to-wakeup,sleep-wakeup-count,sleep-heartrate-mean,sleep-heartrate-min,sleep-heartrate-max,step-count,step-ratio,step-mean,step-max,step-max-timestamp
0,1,2022-03-31,2022-04-13,5,5,4,4,5,2,25,...,0.00,3,59,54,74,415,0.5000,34.5833,106,13
1,1,2022-04-01,2022-04-13,5,5,4,4,5,2,25,...,0.00,4,61,56,82,447,0.2917,63.8571,185,8
2,1,2022-04-02,2022-04-13,5,5,4,4,5,2,25,...,0.17,3,59,55,83,533,0.4231,56.1132,170,14


In [6]:
df = data[["sis", "ohs", "oks"]]

In [7]:
X = data.iloc[:, -46:]   ### selezioni i predictors

In [8]:
# =============================================================================
# # Compute correlation matrices
correlation_matrix_pearson = df.corr(method='pearson')  # Linear relationships
correlation_matrix_spearman = df.corr(method='spearman')  # Rank-based relationships
correlation_matrix_kendall = df.corr(method='kendall')  # Ordinal relationships

In [9]:
# Function to plot correlation matrix
def save_correlation_matrix(corr_matrix, title, filename):
     plt.figure(figsize=(8, 6))
     sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
     plt.title(title)
     # Save the figure as pdf
     plt.savefig(filename, bbox_inches='tight')
     plt.close()

In [10]:
# # Save heatmaps as images
save_correlation_matrix(correlation_matrix_pearson, "Pearson Correlation Matrix", "results/pearson_correlation.pdf")
save_correlation_matrix(correlation_matrix_spearman, "Spearman Correlation Matrix", "results/spearman_correlation.pdf")
save_correlation_matrix(correlation_matrix_kendall, "Kendall Correlation Matrix", "results/kendall_correlation.pdf")

In [11]:
# # Print results
print("\nPearson Correlation Matrix:\n", correlation_matrix_pearson)
print("\nSpearman Correlation Matrix:\n", correlation_matrix_spearman)
print("\nKendall Correlation Matrix:\n", correlation_matrix_kendall)


Pearson Correlation Matrix:
           sis       ohs       oks
sis  1.000000  0.072648 -0.037319
ohs  0.072648  1.000000  0.328511
oks -0.037319  0.328511  1.000000

Spearman Correlation Matrix:
           sis       ohs       oks
sis  1.000000  0.131743 -0.042435
ohs  0.131743  1.000000  0.267941
oks -0.042435  0.267941  1.000000

Kendall Correlation Matrix:
           sis       ohs       oks
sis  1.000000  0.097828 -0.030881
ohs  0.097828  1.000000  0.205677
oks -0.030881  0.205677  1.000000


In [12]:
# # Compute quartiles for SISS, OHSS, and OKSS
siss_q1, siss_q2, siss_q3 = np.percentile(df["sis"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(df["ohs"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(df["oks"], [25, 50, 75])

In [13]:
# # Define quartile-based bins and labels
quartile_labels = ["Q1 (Lowest)", "Q2", "Q3", "Q4 (Highest)"]

In [14]:
# # Discretization for SISS
siss_bins = [df["sis"].min(), siss_q1, siss_q2, siss_q3, df["sis"].max()]
df["SISS_Category_Q"] = pd.cut(df["sis"], bins=siss_bins, labels=quartile_labels, include_lowest=True)

In [15]:
# # Discretization for OHSS
ohss_bins = [df["ohs"].min(), ohss_q1, ohss_q2, ohss_q3, df["ohs"].max()]
df["OHSS_Category_Q"] = pd.cut(df["ohs"], bins=ohss_bins, labels=quartile_labels, include_lowest=True)

In [16]:
# # Discretization for OKSS
okss_bins = [df["oks"].min(), okss_q1, okss_q2, okss_q3, df["oks"].max()]
df["OKSS_Category_Q"] = pd.cut(df["oks"], bins=okss_bins, labels=quartile_labels, include_lowest=True)

In [17]:
# # Compute frequency distributions
siss_freq_q = df["SISS_Category_Q"].value_counts().sort_index()
ohss_freq_q = df["OHSS_Category_Q"].value_counts().sort_index()
okss_freq_q = df["OKSS_Category_Q"].value_counts().sort_index()

In [18]:
# # Print frequency distributions
print("\nSISS Quartile Frequency Distribution:\n", siss_freq_q)
print("\nOHSS Quartile Frequency Distribution:\n", ohss_freq_q)
print("\nOKSS Quartile Frequency Distribution:\n", okss_freq_q)


SISS Quartile Frequency Distribution:
 SISS_Category_Q
Q1 (Lowest)     252
Q2              266
Q3              294
Q4 (Highest)    196
Name: count, dtype: int64

OHSS Quartile Frequency Distribution:
 OHSS_Category_Q
Q1 (Lowest)     266
Q2              392
Q3               98
Q4 (Highest)    252
Name: count, dtype: int64

OKSS Quartile Frequency Distribution:
 OKSS_Category_Q
Q1 (Lowest)     252
Q2              252
Q3              252
Q4 (Highest)    252
Name: count, dtype: int64


In [19]:
# # Plot frequency distributions
def save_quartile_distribution(freq_data, title, filename):
     plt.figure(figsize=(8, 5))
     sns.barplot(x=freq_data.index, y=freq_data.values, palette="coolwarm")
     plt.xlabel("Quartile")
     plt.ylabel("Frequency")
     plt.title(title)
     plt.xticks(rotation=45)
     plt.savefig(filename, bbox_inches='tight')
     plt.close()

In [20]:
# # Generate plots
save_quartile_distribution(siss_freq_q, "Frequency Distribution of SISS Quartiles", "results/quartile_siss.pdf")
save_quartile_distribution(ohss_freq_q, "Frequency Distribution of OHSS Quartiles", "results/quartile_ohss.pdf")
save_quartile_distribution(okss_freq_q, "Frequency Distribution of OKSS Quartiles", "results/quartile_okss.pdf")

In [21]:
# Function to create and save boxplots for each discretized variable
def save_boxplot(data, feature, title, filename):
    plt.figure(figsize=(8, 5))
    sns.boxplot(y=data[feature], palette="coolwarm")
    plt.ylabel(feature)
    plt.title(title)
    plt.xticks(rotation=45)
    
    # Save the figure as PNG
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()

In [23]:
# Save boxplots for each discretized variable
save_boxplot(df, "sis", "SISS box plot", "results/siss_boxplot.pdf")
save_boxplot(df, "ohs",  "OHSS box plot", "results/ohss_boxplot.pdf")
save_boxplot(df, "oks", "OKSS box plot", "results/okss_boxplot.pdf")

In [24]:
for q, name in [(siss_q1,"Q1"), (siss_q2,"Q2"), (siss_q3,"Q3")]:
    print(name, q, (df["sis"] == q).sum())

Q1 21.75 0
Q2 24.0 112
Q3 27.0 98


In [25]:
df["SISS_Category_Q"] = pd.qcut(df["sis"], q=4, labels=quartile_labels)
df["SISS_Category_Q"].value_counts().sort_index()

SISS_Category_Q
Q1 (Lowest)     252
Q2              266
Q3              294
Q4 (Highest)    196
Name: count, dtype: int64